In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "scikit-learn", "bitsandbytes", "huggingface_hub", "pandas"], check=False)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'tokenizers', 'scikit-learn', 'bitsandbytes', 'huggingface_hub', 'pandas'], returncode=0)

In [2]:
import os, re, json, time, math, random, zipfile, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.float16 if DEVICE == "cuda" else torch.bfloat16
WORK, RAW_DIR = "/kaggle/working", "/kaggle/temp/hf_data"
OUT_DIR = os.path.join(WORK, "phase5")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(RAW_DIR, exist_ok=True)

# where the Phase 1 checkpoints (A_decoder.pt ...) and bpe32k.json live (add them as a Kaggle input dataset)
CKPT_ROOTS = ["/kaggle/input", "/kaggle/working"]

# long-document settings: CH tokens per chunk (Phase 1 trained on 1024), up to MAX_CHUNKS chunks per doc
CFG = dict(CH=512, STRIDE=384, MAX_CHUNKS=2, MODE="spread",   # MODE: head | spread | headtail
           BS=4, ACCUM=2, POOL="mean", MAX_TOKENS=60000)      # POOL: mean | max | attn

def find_file(fname):
    for root in CKPT_ROOTS:
        hits = sorted(glob.glob(os.path.join(root, "**", "*" + fname), recursive=True))
        hits = [h for h in hits if "_last" not in os.path.basename(h)]
        if hits: return hits[0]
    return None
print(DEVICE, torch.__version__)

cuda 2.11.0+cu128


In [3]:
from huggingface_hub import snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")     # store your NEW token as a Kaggle secret
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

snapshot_download(repo_id="Exploration-Lab/CS-781-Capstone", repo_type="dataset", local_dir=RAW_DIR,
                  token=HF_TOKEN, allow_patterns=["Phase_5_LSI/*", "Phase_5_LSI/**"])
for dp, _, fns in os.walk(RAW_DIR):
    for fn in fns:
        p = os.path.join(dp, fn); print(f"{os.path.getsize(p)/1e6:9.1f} MB  {p}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

     79.4 MB  /kaggle/temp/hf_data/Phase_5_LSI/Train/dev.json
    299.4 MB  /kaggle/temp/hf_data/Phase_5_LSI/Train/train.json
     98.1 MB  /kaggle/temp/hf_data/Phase_5_LSI/Test/test.json
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/CACHEDIR.TAG
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/.gitignore
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/download/Phase_5_LSI/Train/dev.json.lock
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/download/Phase_5_LSI/Train/train.json.metadata
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/download/Phase_5_LSI/Train/dev.json.metadata
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/download/Phase_5_LSI/Train/train.json.lock
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/download/Phase_5_LSI/Test/test.json.lock
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/download/Phase_5_LSI/Test/test.json.metadata
      0.0 MB  /kaggle/temp/hf_data/.cache/huggingface/trees/93b0059be756c6c461f3a225de25aabda7

In [4]:
def locate(fn):
    hits = [h for h in glob.glob(os.path.join(RAW_DIR, "**", fn), recursive=True) if "Phase_5" in h]
    assert hits, f"{fn} not found"
    return hits[0]

def load_json(p):
    d = json.load(open(p, encoding="utf-8"))
    if isinstance(d, dict):                       # {id: {...}} style
        d = [{"id": k, **(v if isinstance(v, dict) else {"text": v})} for k, v in d.items()]
    return d

train = load_json(locate("train.json"))
dev   = load_json(locate("dev.json"))
test  = load_json(locate("test.json"))
print(len(train), len(dev), len(test), "| keys:", list(train[0].keys()))
print("labels/doc (train):", np.mean([len(d["labels"]) for d in train]).round(2))

42750 10181 13019 | keys: ['id', 'text', 'labels']
labels/doc (train): 3.71


In [5]:
TOK_PATH = globals().get("TOK_PATH") or find_file("bpe32k.json")
assert TOK_PATH, "bpe32k.json not found - add your Phase 1 outputs as a Kaggle dataset"
tok = Tokenizer.from_file(TOK_PATH)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = [tok.token_to_id(s) for s in ["<pad>", "<unk>", "<bos>", "<eos>"]]
VOCAB_SIZE = tok.get_vocab_size()
print(TOK_PATH, VOCAB_SIZE, PAD_ID, EOS_ID)

/kaggle/input/datasets/samswe/bpefile/bpe32k.json 32000 0 3


In [6]:
import math
# ===== Phase 1 model definitions (verbatim from the Phase 1 notebook) =====
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

def rotary_freqs(dim, max_seq, base=10000.0, device="cpu"):
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_seq, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rotary(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., :d // 2], x[..., d // 2:]
    cos = cos[:x.shape[-2], :d // 2].unsqueeze(0).unsqueeze(0)
    sin = sin[:x.shape[-2], :d // 2].unsqueeze(0).unsqueeze(0)
    rx1 = x1 * cos - x2 * sin
    rx2 = x2 * cos + x1 * sin
    return torch.cat([rx1, rx2], dim=-1)

class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))
print("blocks defined")

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.n_heads, self.hd = n_heads, dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        cos, sin = rotary_freqs(self.hd, max_seq)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.hd).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = apply_rotary(q, self.cos, self.sin), apply_rotary(k, self.cos, self.sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.proj(out)

class DecoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq, mlp_mult=4):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, max_seq)
        self.norm2 = RMSNorm(dim)
        self.mlp = SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, dim=768, n_layers=12, n_heads=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([DecoderBlock(dim, n_heads, max_seq) for _ in range(n_layers)])
        self.norm_f = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight
        self.max_seq = max_seq
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for b in self.blocks:
            x = b(x)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=PAD_ID)
        return logits, loss

def count_params(m):
    return sum(p.numel() for p in m.parameters())


class Attn(nn.Module):
    def __init__(self, dim, n_heads, max_seq, n_kv_heads=None, causal=True, window=None, rope=True):
        super().__init__()
        self.h, self.kv = n_heads, (n_kv_heads or n_heads)
        self.hd = dim // n_heads
        self.causal, self.window, self.rope = causal, window, rope
        self.q = nn.Linear(dim, self.h * self.hd, bias=False)
        self.k = nn.Linear(dim, self.kv * self.hd, bias=False)
        self.v = nn.Linear(dim, self.kv * self.hd, bias=False)
        self.proj = nn.Linear(self.h * self.hd, dim, bias=False)
        if rope:
            cos, sin = rotary_freqs(self.hd, max_seq)
            self.register_buffer("cos", cos, persistent=False)
            self.register_buffer("sin", sin, persistent=False)

    def _mask(self, T, device):
        i = torch.arange(T, device=device)[:, None]
        j = torch.arange(T, device=device)[None, :]
        return (j <= i) & (i - j < self.window)

    def forward(self, x, ctx=None):
        B, T, C = x.shape
        src = x if ctx is None else ctx
        S = src.size(1)
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k(src).view(B, S, self.kv, self.hd).transpose(1, 2)
        v = self.v(src).view(B, S, self.kv, self.hd).transpose(1, 2)
        if self.rope and ctx is None:
            q, k = apply_rotary(q, self.cos, self.sin), apply_rotary(k, self.cos, self.sin)
        if self.kv != self.h:
            r = self.h // self.kv
            k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
        if ctx is None and self.causal and self.window:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=self._mask(T, x.device))
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=(self.causal and ctx is None))
        return self.proj(out.transpose(1, 2).reshape(B, T, self.h * self.hd))
print("Attn defined")

class EncBlock(nn.Module):
    def __init__(self, dim, nh, ms, mlp_mult=4):
        super().__init__()
        self.n1, self.attn = RMSNorm(dim), Attn(dim, nh, ms, causal=False)
        self.n2, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.mlp(self.n2(x))

class CrossDecBlock(nn.Module):
    def __init__(self, dim, nh, ms, mlp_mult=4):
        super().__init__()
        self.n1, self.self_attn = RMSNorm(dim), Attn(dim, nh, ms, causal=True)
        self.n2, self.cross = RMSNorm(dim), Attn(dim, nh, ms, causal=False, rope=False)
        self.n3, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x, enc):
        x = x + self.self_attn(self.n1(x))
        x = x + self.cross(self.n2(x), ctx=enc)
        return x + self.mlp(self.n3(x))

class EncoderDecoderLM(nn.Module):
    """Same call as the decoder-only model: forward(idx, targets).
    Encoder sees idx[:, :split]; decoder predicts idx[:, split:]."""
    def __init__(self, vocab=VOCAB_SIZE, dim=768, n_enc=6, n_dec=6, n_heads=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.enc = nn.ModuleList([EncBlock(dim, n_heads, max_seq) for _ in range(n_enc)])
        self.dec = nn.ModuleList([CrossDecBlock(dim, n_heads, max_seq) for _ in range(n_dec)])
        self.norm_e, self.norm_f = RMSNorm(dim), RMSNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight
        self.max_seq = max_seq
    def encode(self, src):
        e = self.tok_emb(src)
        for b in self.enc: e = b(e)
        return self.norm_e(e)
    def decode(self, dec_in, enc_out):
        x = self.tok_emb(dec_in)
        for b in self.dec: x = b(x, enc_out)
        return self.head(self.norm_f(x))
    def forward(self, idx, targets=None, split=None):
        T = idx.size(1)
        split = split or T // 2
        enc_out = self.encode(idx[:, :split])
        logits = self.decode(idx[:, split - 1:], enc_out)     # decoder starts from last prompt token
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   targets[:, split - 1:].reshape(-1), ignore_index=PAD_ID)
        return logits, loss
print("EncoderDecoderLM defined")

class GatedLinearRecurrence(nn.Module):
    """h_t = a_t * h_{t-1} + i_t * u_t, computed in parallel by chunks (linear time)."""
    def __init__(self, dim, chunk=64):
        super().__init__()
        self.inp = nn.Linear(dim, 4 * dim, bias=False)
        self.conv = nn.Conv1d(dim, dim, 4, groups=dim, padding=3)
        self.out = nn.Linear(dim, dim, bias=False)
        self.chunk = chunk
    def _scan(self, log_a, z):
        B, T, C = z.shape
        L = self.chunk
        pad = (-T) % L
        if pad:
            log_a, z = F.pad(log_a, (0, 0, 0, pad)), F.pad(z, (0, 0, 0, pad))
        n = (T + pad) // L
        log_a, z = log_a.reshape(B, n, L, C), z.reshape(B, n, L, C)
        A = log_a.cumsum(2)
        local = torch.exp(A) * torch.cumsum(torch.exp(-A) * z, 2)
        prev, outs = z.new_zeros(B, C), []
        for j in range(n):
            o = local[:, j] + torch.exp(A[:, j]) * prev[:, None]
            prev = o[:, -1]
            outs.append(o)
        return torch.stack(outs, 1).reshape(B, n * L, C)[:, :T]
    def forward(self, x):
        B, T, C = x.shape
        u, ga, gi, go = self.inp(x).chunk(4, -1)
        u = self.conv(u.transpose(1, 2))[..., :T].transpose(1, 2)
        log_a = -0.05 * torch.sigmoid(ga.float())          # decay in (e^-0.05, 1)
        z = (torch.sigmoid(gi) * F.silu(u)).float()
        h = self._scan(log_a, z).to(x.dtype)
        return self.out(h * F.silu(go))

class Block(nn.Module):
    def __init__(self, dim, mixer, mlp_mult=4):
        super().__init__()
        self.norm1, self.mixer = RMSNorm(dim), mixer
        self.norm2, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class GenericLM(nn.Module):
    def __init__(self, make_block, vocab=VOCAB_SIZE, dim=768, n_layers=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([make_block(i) for i in range(n_layers)])
        self.norm_f = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight
        self.max_seq = max_seq
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for b in self.blocks: x = b(x)
        logits = self.head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=PAD_ID)
        return logits, loss

def make_hybrid(dim=768, n_heads=12, max_seq=2048, attn_every=3):
    def f(i):
        mixer = Attn(dim, n_heads, max_seq) if (i + 1) % attn_every == 0 else GatedLinearRecurrence(dim)
        return Block(dim, mixer)
    return f
print("Hybrid pieces defined")

def make_efficient(dim=768, n_heads=12, n_kv=4, window=256, global_every=4, max_seq=2048):
    def f(i):
        is_global = (i + 1) % global_every == 0
        mixer = Attn(dim, n_heads, max_seq, n_kv_heads=n_kv, window=None if is_global else window)
        return Block(dim, mixer, mlp_mult=3)
    return f
print("Efficient builder defined")

def init_weights(m, n_layers=12):
    for mod in m.modules():
        if isinstance(mod, (nn.Linear, nn.Embedding)):
            nn.init.normal_(mod.weight, mean=0.0, std=0.02)
    # scale down the layers that write into the residual stream (GPT-2 style)
    for name, p in m.named_parameters():
        if name.endswith(("proj.weight", "w3.weight", "out.weight")):
            nn.init.normal_(p, mean=0.0, std=0.02 / (2 * n_layers) ** 0.5)
    return m

def build(name):
    if name == "A_decoder":   m = DecoderOnlyLM(dim=768, n_layers=12, n_heads=12)
    if name == "B_encdec":    m = EncoderDecoderLM(dim=768, n_enc=6, n_dec=6, n_heads=12)
    if name == "C_hybrid":    m = GenericLM(make_hybrid(), dim=768, n_layers=12)
    if name == "D_efficient": m = GenericLM(make_efficient(), dim=768, n_layers=12)
    return init_weights(m)

blocks defined
Attn defined
EncoderDecoderLM defined
Hybrid pieces defined
Efficient builder defined


In [7]:
all_labels = sorted({int(l) for d in train + dev for l in d["labels"]})
L = len(all_labels); lab2i = {l: i for i, l in enumerate(all_labels)}
def multihot(docs):
    Y = np.zeros((len(docs), L), np.float32)
    for i, d in enumerate(docs):
        for l in d["labels"]:
            if int(l) in lab2i: Y[i, lab2i[int(l)]] = 1
    return Y
Ytr, Yd = multihot(train), multihot(dev)
print("num labels:", L, "| train support min/median/max:", int(Ytr.sum(0).min()), int(np.median(Ytr.sum(0))), int(Ytr.sum(0).max()))

def tokenize_docs(docs):
    out = []
    for i in tqdm(range(0, len(docs), 256), desc="tokenize"):
        enc = tok.encode_batch([d["text"] for d in docs[i:i + 256]])
        out += [np.array(e.ids[:CFG["MAX_TOKENS"]], dtype=np.int32) for e in enc]
    return out
ids_tr, ids_dv, ids_te = tokenize_docs(train), tokenize_docs(dev), tokenize_docs(test)
print("tokens/doc (train) mean/median/max:", int(np.mean([len(a) for a in ids_tr])), int(np.median([len(a) for a in ids_tr])), max(len(a) for a in ids_tr))

def chunk_starts(n, CH, stride, k, mode):
    if n <= CH: return [0]
    st = list(range(0, n - CH + 1, stride))
    if st[-1] != n - CH: st.append(n - CH)
    if len(st) <= k: return st
    if mode == "head": return st[:k]
    if mode == "headtail": h = k // 2; return st[:h] + st[-(k - h):]
    sel = sorted(set(np.linspace(0, len(st) - 1, k).round().astype(int)))     # "spread": evenly across the document
    return [st[i] for i in sel]

class DocDS(torch.utils.data.Dataset):
    def __init__(self, ids, Y=None): self.ids, self.Y = ids, Y
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        a, CH = self.ids[i], CFG["CH"]
        starts = chunk_starts(len(a), CH, CFG["STRIDE"], CFG["MAX_CHUNKS"], CFG["MODE"])
        ch = np.full((len(starts), CH), PAD_ID, np.int64); m = np.zeros((len(starts), CH), bool)
        for j, s in enumerate(starts):
            seg = a[s:s + CH]; ch[j, :len(seg)] = seg; m[j, :len(seg)] = True
        y = self.Y[i] if self.Y is not None else np.zeros(L, np.float32)
        return i, ch, m, y

def collate(b):
    return (torch.tensor([x[0] for x in b]), torch.from_numpy(np.concatenate([x[1] for x in b])),
            torch.from_numpy(np.concatenate([x[2] for x in b])), [len(x[1]) for x in b],
            torch.tensor(np.stack([x[3] for x in b]), dtype=torch.float32))

tr_ds, dv_ds, te_ds = DocDS(ids_tr, Ytr), DocDS(ids_dv, Yd), DocDS(ids_te)

num labels: 100 | train support min/median/max: 272 679 11730


tokenize:   0%|          | 0/167 [00:00<?, ?it/s]

tokenize:   0%|          | 0/40 [00:00<?, ?it/s]

tokenize:   0%|          | 0/51 [00:00<?, ?it/s]

tokens/doc (train) mean/median/max: 1751 662 60000


In [8]:
def find_ckpt(name): 
    p = find_file(name + ".pt"); assert p, f"checkpoint for {name} not found"; return p

def load_backbone(name):
    m = build(name)
    sd = torch.load(find_ckpt(name), map_location="cpu")
    if isinstance(sd.get("model"), dict): sd = sd["model"]
    sd = {k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()}
    print(name, m.load_state_dict(sd, strict=True))
    if name == "B_encdec": m.dec = nn.ModuleList()          # only the encoder is used for classification
    return m

def blocks_of(m, name): return m.enc if name == "B_encdec" else m.blocks

def hidden_states(m, name, idx):
    if name == "B_encdec": return m.encode(idx)             # bidirectional encoder, already normed
    x = m.tok_emb(idx)
    for b in m.blocks: x = b(x)
    return m.norm_f(x)

class DocClassifier(nn.Module):
    # chunks -> masked-mean token pooling -> chunk vectors -> doc pooling (mean/max/attn) -> multi-label head
    def __init__(self, backbone, name, n_labels, pool="mean", dim=768, drop=0.1):
        super().__init__()
        self.bb, self.name, self.pool = backbone, name, pool
        self.attn = nn.Linear(dim, 1) if pool == "attn" else None
        self.drop, self.head = nn.Dropout(drop), nn.Linear(dim, n_labels)
    def forward(self, chunks, mask, counts):
        h = hidden_states(self.bb, self.name, chunks).float()
        mk = mask.unsqueeze(-1).float()
        v = (h * mk).sum(1) / mk.sum(1).clamp(min=1)
        docs = []
        for part in torch.split(v, counts):
            if self.pool == "mean": d = part.mean(0)
            elif self.pool == "max": d = part.max(0).values
            else:
                w = torch.softmax(self.attn(part).squeeze(-1), 0); d = (w.unsqueeze(-1) * part).sum(0)
            docs.append(d)
        emb = torch.stack(docs)
        return self.head(self.drop(emb)), emb

In [9]:
TARGETS = ("q", "k", "v", "qkv", "proj", "w1", "w2", "w3", "inp", "out")

class LoRALinear(nn.Module):                    # additive:        W' = W0 + s * B A
    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x)
        return y + (((x @ self.A.t()) @ self.B.t()) * self.scale).to(y.dtype)

class LoRMALinear(nn.Module):                   # multiplicative:  W' = (I + B A) W0   ->  y = h + B A h,  h = W0 x
    def __init__(self, base, r=16):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        h = self.base(x)
        return h + ((h @ self.A.t()) @ self.B.t()).to(h.dtype)

class AdapterMLP(nn.Module):                    # bottleneck adapter after the MLP
    def __init__(self, mlp, dim=768, r=64):
        super().__init__()
        self.mlp, self.down, self.up = mlp, nn.Linear(dim, r), nn.Linear(r, dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.mlp(x); return y + self.up(F.gelu(self.down(y)))

def to_nf4(lin):
    import bitsandbytes as bnb
    q = bnb.nn.Linear4bit(lin.in_features, lin.out_features, bias=False, compute_dtype=torch.float16, quant_type="nf4")
    q.weight = bnb.nn.Params4bit(lin.weight.data.half().cpu(), requires_grad=False, quant_type="nf4")
    return q                                    # actually quantized when the model is moved to the GPU

def wrap_linears(mod, factory):
    for n, c in list(mod.named_children()):
        if isinstance(c, nn.Linear) and n in TARGETS: setattr(mod, n, factory(c))
        else: wrap_linears(c, factory)

def apply_method(bb, name, method, r=16, k_last=4):
    blocks = blocks_of(bb, name)
    for p in bb.parameters(): p.requires_grad = False
    if method == "full":
        for p in bb.parameters(): p.requires_grad = True
    elif method == "partial":                   # last k blocks + final norm
        for b in blocks[-k_last:]:
            for p in b.parameters(): p.requires_grad = True
        for p in (bb.norm_e if name == "B_encdec" else bb.norm_f).parameters(): p.requires_grad = True
    elif method == "lora":   wrap_linears(blocks, lambda c: LoRALinear(c, r, 2 * r))
    elif method == "lorma":  wrap_linears(blocks, lambda c: LoRMALinear(c, r))
    elif method == "qlora":  wrap_linears(blocks, lambda c: LoRALinear(to_nf4(c), r, 2 * r))
    elif method == "adapter":
        for b in blocks: b.mlp = AdapterMLP(b.mlp)
    else: raise ValueError(method)
    return bb

In [10]:
from sklearn.metrics import f1_score

def make_hard_loss(kind, Y):
    pos = Y.sum(0); neg = len(Y) - pos
    pw = torch.tensor(np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1, 30), dtype=torch.float32, device=DEVICE)
    if kind == "bce":  return lambda z, y: F.binary_cross_entropy_with_logits(z, y)
    if kind == "wbce": return lambda z, y: F.binary_cross_entropy_with_logits(z, y, pos_weight=pw)
    if kind == "focal":
        def f(z, y, g=2.0):
            p = torch.sigmoid(z); ce = F.binary_cross_entropy_with_logits(z, y, reduction="none")
            return (((1 - (p * y + (1 - p) * (1 - y))) ** g) * ce).mean()
        return f
    raise ValueError(kind)

def decode(P, thr):                             # probs -> multi-hot; at least one label per document
    out = (P >= thr).astype(np.int32)
    e = out.sum(1) == 0
    if e.any(): out[e, P[e].argmax(1)] = 1
    return out

def metrics(Y, Pm):
    Y = Y.astype(int)
    return dict(macro_f1=f1_score(Y, Pm, average="macro", zero_division=0),
                micro_f1=f1_score(Y, Pm, average="micro", zero_division=0),
                subset_acc=float((Y == Pm).all(1).mean()))

GRID = np.arange(0.05, 0.951, 0.025)
def tune_global(Y, P):
    best = (0.5, -1.0)
    for t in GRID:
        Pm = decode(P, t)
        tp = (Pm * Y).sum(0); fp = (Pm * (1 - Y)).sum(0); fn = ((1 - Pm) * Y).sum(0)
        f = float((2 * tp / np.maximum(2 * tp + fp + fn, 1)).mean())
        if f > best[1]: best = (float(t), f)
    return best

def tune_per_label(Y, P, thr_g, min_support=3, shrink=0.5):
    thr = np.full(Y.shape[1], thr_g, dtype=np.float32)
    for j in range(Y.shape[1]):
        if Y[:, j].sum() < min_support: continue            # rare on dev: keep the global threshold
        pred = P[:, j][None, :] >= GRID[:, None]; yj = (Y[:, j] > 0.5)[None, :]
        tp = (pred & yj).sum(1); fp = (pred & ~yj).sum(1); fn = ((~pred) & yj).sum(1)
        f = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        thr[j] = shrink * GRID[f.argmax()] + (1 - shrink) * thr_g
    return thr

In [11]:
RESULTS = {}

@torch.no_grad()
def infer(model, ds, bs=None):
    model.eval()
    dl = torch.utils.data.DataLoader(ds, batch_size=bs or CFG["BS"] * 2, shuffle=False, collate_fn=collate, num_workers=2)
    Z, E = [], []
    for idx, ch, m, counts, _ in dl:
        with torch.autocast(DEVICE, dtype=AMP_DTYPE):
            z, e = model(ch.to(DEVICE), m.to(DEVICE), counts)
        Z.append(z.float().cpu()); E.append(e.float().cpu())
    return torch.cat(Z), torch.cat(E)

def run_experiment(tag, name, method, loss_kind="wbce", epochs=5, lr_bb=5e-5, lr_head=1e-3, r=16,
                   backbone=None, kd=None, save_train_out=False, pool=None):
    pool = pool or CFG["POOL"]
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
    bb = backbone if backbone is not None else load_backbone(name)
    apply_method(bb, name, method, r)
    model = DocClassifier(bb, name, L, pool).to(DEVICE)
    n_total = sum(p.numel() for p in model.parameters()); n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{tag}] {name}/{method}  params {n_total/1e6:.1f}M  trainable {n_train/1e6:.2f}M ({100*n_train/n_total:.1f}%)")

    hard = make_hard_loss(loss_kind, Ytr)
    hd = list(model.head.parameters()) + (list(model.attn.parameters()) if model.attn is not None else [])
    groups = [{"params": hd, "lr": lr_head}]
    bbp = [p for p in model.bb.parameters() if p.requires_grad]
    if bbp: groups.append({"params": bbp, "lr": lr_bb})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    for g in opt.param_groups: g["base"] = g["lr"]
    tr_dl = torch.utils.data.DataLoader(tr_ds, batch_size=CFG["BS"], shuffle=True, collate_fn=collate, num_workers=2)
    total = epochs * math.ceil(len(tr_dl) / CFG["ACCUM"]); warm = max(1, int(0.1 * total))
    def set_lr(s):
        f = (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
        for g in opt.param_groups: g["lr"] = g["base"] * f
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    best, step, t0 = dict(f1=-1), 0, time.time()
    for ep in range(epochs):
        model.train(); run, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for it, (idx, ch, m, counts, y) in enumerate(tr_dl):
            if it % CFG["ACCUM"] == 0: set_lr(step)
            with torch.autocast(DEVICE, dtype=AMP_DTYPE):
                z, e = model(ch.to(DEVICE), m.to(DEVICE), counts)
            z, y = z.float(), y.to(DEVICE)
            loss = hard(z, y)
            if kd:                              # knowledge distillation: soft labels + embedding matching
                zt, et, T = kd["logits"][idx].to(DEVICE), kd["emb"][idx].to(DEVICE), kd["T"]
                soft = F.binary_cross_entropy_with_logits(z / T, torch.sigmoid(zt / T)) * T * T
                loss = kd["alpha"] * loss + (1 - kd["alpha"]) * soft + kd["beta"] * F.mse_loss(e.float(), et)
            scaler.scale(loss / CFG["ACCUM"]).backward()
            run += loss.item(); nb += 1
            if (it + 1) % CFG["ACCUM"] == 0 or it + 1 == len(tr_dl):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        Zd, _ = infer(model, dv_ds); Pd = torch.sigmoid(Zd).numpy()
        thr, f1 = tune_global(Yd, Pd)
        print(f"  ep {ep+1}/{epochs} loss {run/nb:.4f} | dev macro-F1 {f1:.4f} @thr {thr:.3f} | {(time.time()-t0)/60:.1f} min", flush=True)
        if f1 > best["f1"]:
            best = dict(f1=f1, ep=ep + 1, probs=Pd,
                        state={k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad})
    with torch.no_grad():
        for k, p in model.named_parameters():
            if k in best["state"]: p.copy_(best["state"][k].to(p.device))
    t1 = time.time(); Zt, _ = infer(model, te_ds); speed = len(te_ds) / (time.time() - t1)
    Pd, Pt = best["probs"], torch.sigmoid(Zt).numpy()
    thr_g, _ = tune_global(Yd, Pd); thr_l = tune_per_label(Yd, Pd, thr_g)
    res = dict(model=name, method=method, loss=loss_kind, pool=pool, mode=CFG["MODE"], best_epoch=best["ep"], thr_global=thr_g,
               **{f"dev_{k}_thr0.5": v for k, v in metrics(Yd, decode(Pd, 0.5)).items()},
               **{f"dev_{k}_global": v for k, v in metrics(Yd, decode(Pd, thr_g)).items()},
               **{f"dev_{k}_label": v for k, v in metrics(Yd, decode(Pd, thr_l)).items()},
               params_M=n_total / 1e6, trainable_M=n_train / 1e6, docs_per_s=speed,
               peak_gpu_GB=(torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0))
    RESULTS[tag] = res
    np.save(f"{OUT_DIR}/{tag}_dev_probs.npy", Pd); np.save(f"{OUT_DIR}/{tag}_test_probs.npy", Pt)
    np.save(f"{OUT_DIR}/{tag}_thr_label.npy", thr_l)
    torch.save(best["state"], f"{OUT_DIR}/{tag}_trainable.pt")
    if save_train_out:
        Ztr, Etr = infer(model, tr_ds); torch.save(dict(logits=Ztr, emb=Etr), f"{OUT_DIR}/{tag}_train_out.pt")
    json.dump(RESULTS, open(f"{OUT_DIR}/results.json", "w"), indent=1)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()})
    del model, bb, opt, best
    if DEVICE == "cuda": torch.cuda.empty_cache()

In [12]:
# Experiment 1 - A_decoder, FULL fine-tuning (also the teacher for distillation)
run_experiment("exp1_A_full", "A_decoder", "full", loss_kind="wbce", epochs=2, lr_bb=5e-5, save_train_out=True)

A_decoder <All keys matched successfully>


[exp1_A_full] A_decoder/full  params 109.6M  trainable 109.61M (100.0%)


  ep 1/2 loss 0.3377 | dev macro-F1 0.3078 @thr 0.450 | 40.6 min


  ep 2/2 loss 0.2516 | dev macro-F1 0.3545 @thr 0.525 | 81.3 min


{'model': 'A_decoder', 'method': 'full', 'loss': 'wbce', 'pool': 'mean', 'mode': 'spread', 'best_epoch': 2, 'thr_global': 0.525, 'dev_macro_f1_thr0.5': 0.3526, 'dev_micro_f1_thr0.5': 0.452, 'dev_subset_acc_thr0.5': 0.0334, 'dev_macro_f1_global': 0.3545, 'dev_micro_f1_global': 0.4529, 'dev_subset_acc_global': 0.0364, 'dev_macro_f1_label': 0.3632, 'dev_micro_f1_label': 0.4589, 'dev_subset_acc_label': 0.0359, 'params_M': 109.6068, 'trainable_M': 109.6068, 'docs_per_s': 64.8187, 'peak_gpu_GB': 4.1795}


In [13]:
# Experiment 2 - B_encdec, LoRMA (compulsory)
run_experiment("exp2_B_lorma", "B_encdec", "lorma", loss_kind="wbce", epochs=3, lr_bb=3e-4, r=16)

B_encdec <All keys matched successfully>
[exp2_B_lorma] B_encdec/lorma  params 68.7M  trainable 1.60M (2.3%)


  ep 1/3 loss 0.4163 | dev macro-F1 0.1634 @thr 0.275 | 19.9 min


  ep 2/3 loss 0.3618 | dev macro-F1 0.2088 @thr 0.350 | 39.9 min


  ep 3/3 loss 0.3410 | dev macro-F1 0.2232 @thr 0.375 | 59.9 min


{'model': 'B_encdec', 'method': 'lorma', 'loss': 'wbce', 'pool': 'mean', 'mode': 'spread', 'best_epoch': 3, 'thr_global': 0.375, 'dev_macro_f1_thr0.5': 0.2037, 'dev_micro_f1_thr0.5': 0.3474, 'dev_subset_acc_thr0.5': 0.022, 'dev_macro_f1_global': 0.2232, 'dev_micro_f1_global': 0.3481, 'dev_subset_acc_global': 0.0055, 'dev_macro_f1_label': 0.236, 'dev_micro_f1_label': 0.3458, 'dev_subset_acc_label': 0.0056, 'params_M': 68.6547, 'trainable_M': 1.6006, 'docs_per_s': 100.5514, 'peak_gpu_GB': 1.4867}


In [14]:
# Experiment 3 - C_hybrid, QLoRA (4-bit NF4 base + LoRA)
run_experiment("exp3_C_qlora", "C_hybrid", "qlora", loss_kind="wbce", epochs=3, lr_bb=3e-4, r=16)

C_hybrid <All keys matched successfully>


[exp3_C_qlora] C_hybrid/qlora  params 72.2M  trainable 2.78M (3.8%)


  ep 1/3 loss 0.3461 | dev macro-F1 0.2898 @thr 0.450 | 60.3 min


  ep 2/3 loss 0.2783 | dev macro-F1 0.3389 @thr 0.550 | 120.6 min


  ep 3/3 loss 0.2239 | dev macro-F1 0.3539 @thr 0.525 | 180.8 min


{'model': 'C_hybrid', 'method': 'qlora', 'loss': 'wbce', 'pool': 'mean', 'mode': 'spread', 'best_epoch': 3, 'thr_global': 0.525, 'dev_macro_f1_thr0.5': 0.3524, 'dev_micro_f1_thr0.5': 0.4509, 'dev_subset_acc_thr0.5': 0.0322, 'dev_macro_f1_global': 0.3539, 'dev_micro_f1_global': 0.4517, 'dev_subset_acc_global': 0.0366, 'dev_macro_f1_label': 0.363, 'dev_micro_f1_label': 0.4624, 'dev_subset_acc_label': 0.0379, 'params_M': 72.2328, 'trainable_M': 2.7803, 'docs_per_s': 32.8063, 'peak_gpu_GB': 4.052}


In [15]:
# Experiment 4 - D_efficient, KNOWLEDGE DISTILLATION: teacher = exp1 (A full FT), student = D_efficient with 6 of 12 layers
t = torch.load(f"{OUT_DIR}/exp1_A_full_train_out.pt")
KD = dict(logits=t["logits"], emb=t["emb"], T=2.0, alpha=0.5, beta=1.0)

student = load_backbone("D_efficient")
keep = (1, 3, 5, 7, 9, 11)                      # keeps the 3 global-attention layers (index 3, 7, 11)
student.blocks = nn.ModuleList([student.blocks[i] for i in keep])
run_experiment("exp4_D_kd", "D_efficient", "full", loss_kind="wbce", epochs=3, lr_bb=1e-4, backbone=student, kd=KD)

D_efficient <All keys matched successfully>
[exp4_D_kd] D_efficient/full  params 55.3M  trainable 55.33M (100.0%)


  ep 1/3 loss 1.0740 | dev macro-F1 0.3160 @thr 0.425 | 18.4 min


  ep 2/3 loss 0.9425 | dev macro-F1 0.3487 @thr 0.475 | 36.8 min


  ep 3/3 loss 0.9079 | dev macro-F1 0.3582 @thr 0.500 | 55.2 min


{'model': 'D_efficient', 'method': 'full', 'loss': 'wbce', 'pool': 'mean', 'mode': 'spread', 'best_epoch': 3, 'thr_global': 0.5, 'dev_macro_f1_thr0.5': 0.3582, 'dev_micro_f1_thr0.5': 0.4579, 'dev_subset_acc_thr0.5': 0.0356, 'dev_macro_f1_global': 0.3582, 'dev_micro_f1_global': 0.4579, 'dev_subset_acc_global': 0.0356, 'dev_macro_f1_label': 0.3677, 'dev_micro_f1_label': 0.4637, 'dev_subset_acc_label': 0.035, 'params_M': 55.3337, 'trainable_M': 55.3337, 'docs_per_s': 146.5526, 'peak_gpu_GB': 2.0392}


In [16]:
# Optional ablations (cheap): long-document representation, pooling, loss. Uses LoRMA on A_decoder.
BASE_CFG = dict(CFG)
for kind, val in [("MODE", "head"), ("MODE", "headtail"), ("POOL", "attn"), ("loss", "bce"), ("loss", "focal")]:
    CFG.update(BASE_CFG)
    loss = "wbce"
    if kind == "loss": loss = val
    else: CFG[kind] = val
    run_experiment(f"abl_{kind}_{val}", "A_decoder", "lorma", loss_kind=loss, epochs=3, lr_bb=3e-4)
CFG.update(BASE_CFG)

A_decoder <All keys matched successfully>


[abl_MODE_head] A_decoder/lorma  params 112.7M  trainable 3.12M (2.8%)


  ep 1/3 loss 0.3560 | dev macro-F1 0.2810 @thr 0.425 | 37.8 min


  ep 2/3 loss 0.2937 | dev macro-F1 0.3180 @thr 0.500 | 75.7 min


  ep 3/3 loss 0.2522 | dev macro-F1 0.3342 @thr 0.500 | 113.6 min


{'model': 'A_decoder', 'method': 'lorma', 'loss': 'wbce', 'pool': 'mean', 'mode': 'head', 'best_epoch': 3, 'thr_global': 0.5, 'dev_macro_f1_thr0.5': 0.3342, 'dev_micro_f1_thr0.5': 0.4369, 'dev_subset_acc_thr0.5': 0.0304, 'dev_macro_f1_global': 0.3342, 'dev_micro_f1_global': 0.4369, 'dev_subset_acc_global': 0.0304, 'dev_macro_f1_label': 0.3449, 'dev_micro_f1_label': 0.4462, 'dev_subset_acc_label': 0.0283, 'params_M': 112.6542, 'trainable_M': 3.1243, 'docs_per_s': 53.4243, 'peak_gpu_GB': 3.1324}


In [ ]:
from collections import Counter
cols = ["model", "method", "best_epoch", "dev_macro_f1_thr0.5", "dev_macro_f1_global", "dev_macro_f1_label",
        "dev_micro_f1_label", "dev_subset_acc_label", "params_M", "trainable_M", "peak_gpu_GB", "docs_per_s"]
tab = pd.DataFrame(RESULTS).T[cols]
print(tab.round(4).to_string())

best_tag = max(RESULTS, key=lambda t: RESULTS[t]["dev_macro_f1_label"])
P = np.load(f"{OUT_DIR}/{best_tag}_dev_probs.npy"); thr = np.load(f"{OUT_DIR}/{best_tag}_thr_label.npy")
Pm = decode(P, thr); Yi = Yd.astype(int)
tp = (Pm * Yi).sum(0); fp = (Pm * (1 - Yi)).sum(0); fn = ((1 - Pm) * Yi).sum(0)
f1l = 2 * tp / np.maximum(2 * tp + fp + fn, 1); sup = Ytr.sum(0)
print("\nbest:", best_tag)
for lo, hi in [(0, 5), (5, 20), (20, 1e9)]:
    s = (sup >= lo) & (sup < hi)
    print(f"train support [{lo},{hi}): {s.sum():4d} labels | mean dev F1 {f1l[s].mean() if s.any() else float('nan'):.3f}")

rec = tp / np.maximum(tp + fn, 1)
missed = pd.DataFrame(dict(statute=all_labels, train_support=sup, dev_support=Yi.sum(0), recall=rec, precision=tp / np.maximum(tp + fp, 1)))
print("\nmost missed statutes (dev support >= 3):")
print(missed[missed.dev_support >= 3].sort_values("recall").head(10).round(3).to_string(index=False))

conf = Counter()
for i in range(len(Yi)):
    for a in np.where((Yi[i] == 1) & (Pm[i] == 0))[0]:
        for b in np.where((Yi[i] == 0) & (Pm[i] == 1))[0]: conf[(all_labels[a], all_labels[b])] += 1
print("\ncommon confusions (true-but-missed -> wrongly predicted):", conf.most_common(10))

co = (Ytr.T @ Ytr); np.fill_diagonal(co, 0)
print("avg labels/doc pred vs true:", Pm.sum(1).mean().round(2), Yi.sum(1).mean().round(2), "| label pairs co-occurring in train:", int((co > 0).sum() // 2))

In [ ]:
SUBMIT_TAGS = ["exp1_A_full", "exp2_B_lorma", "exp3_C_qlora", "exp4_D_kd"]
SUB_DIR = os.path.join(WORK, "submissions_phase5"); os.makedirs(SUB_DIR, exist_ok=True)

def make_submission(tag, use="label"):
    Pt = np.load(f"{OUT_DIR}/{tag}_test_probs.npy")
    thr = np.load(f"{OUT_DIR}/{tag}_thr_label.npy") if use == "label" else RESULTS[tag]["thr_global"]
    Pm = decode(Pt, thr)
    sub = [{"id": d["id"], "pred": sorted({int(all_labels[j]) for j in np.where(Pm[i])[0]})} for i, d in enumerate(test)]
    assert len(sub) == len(test) and len({s["id"] for s in sub}) == len(test)
    assert all(len(s["pred"]) >= 1 and all(isinstance(x, int) for x in s["pred"]) for s in sub)
    jp, zp = f"{SUB_DIR}/{tag}_prediction.json", f"{SUB_DIR}/{tag}.zip"
    json.dump(sub, open(jp, "w")); 
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(jp, arcname="prediction.json")
    print(tag, "->", zp, "| avg labels/doc:", np.mean([len(s["pred"]) for s in sub]).round(2), "| sample:", sub[0])

for t in SUBMIT_TAGS: make_submission(t, use="label")     # or use="global" if per-label thresholds overfit dev

In [ ]:
run_experiment("exp5_B_full", "B_encdec", "full", loss_kind="wbce", epochs=2, lr_bb=5e-5)
run_experiment("exp6_C_full", "C_hybrid", "full", loss_kind="wbce", epochs=2, lr_bb=5e-5)

for t in ["exp5_B_full", "exp6_C_full"]:
    make_submission(t, use="label")

In [ ]:
import shutil
res = json.load(open(f"{OUT_DIR}/results.json"))
cand = {t: r for t, r in res.items() if not t.startswith("abl_")}
rank = sorted(cand, key=lambda t: cand[t]["dev_macro_f1_label"], reverse=True)
for t in rank:
    print(f"{t:16s} macro-F1 (per-label thr) {cand[t]['dev_macro_f1_label']:.4f} | global {cand[t]['dev_macro_f1_global']:.4f}")

BEST_DIR = os.path.join(WORK, "best2"); os.makedirs(BEST_DIR, exist_ok=True)
for t in rank[:2]:
    src = f"{SUB_DIR}/{t}.zip"
    if not os.path.exists(src):
        make_submission(t, use="label")     # needs the earlier cells in this session; otherwise rerun the setup and submission cells
    shutil.copy(src, f"{BEST_DIR}/{t}.zip")
print("best two:", rank[:2], "->", BEST_DIR)